In [0]:
import data.utilities.common as Commonlib

import pyspark.sql.functions as F
import yaml

from beertech_utils.data.integrity.integrity_check import EmptyDataFrameCheck
from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.types import TimestampType

In [0]:
CONFIG_BASE_PATH = "configuration/"

config_entries = Commonlib.list_files(CONFIG_BASE_PATH, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "configuration file")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}/{config_file}"

config = Commonlib.get_object_config(config_file_path)

In [0]:
medallion = Medallion(config)

In [0]:
SOURCE_TABLES = {
    # silver source tables
    "rm_deal_hdr": MTU.get_fully_qualified_table_name("oracle", "rmr", "rm_deal_hdr"),
    "rm_deal_dtl": MTU.get_fully_qualified_table_name("oracle", "rmr", "rm_deal_dtl"),
    "rm_var_splts": MTU.get_fully_qualified_table_name("oracle", "rmr", "rm_var_splts"),
    "rm_nrp_vers": MTU.get_fully_qualified_table_name("oracle", "rmr", "rm_nrp_vers"),
    "rmr_stat": MTU.get_fully_qualified_table_name("external", "static", "rmr_stat"),
    "rmr_actn": MTU.get_fully_qualified_table_name("external", "static", "rmr_actn"),
    "promo_cust_typ": MTU.get_fully_qualified_table_name("external", "static", "promo_cust_typ"),
    "qty_disc_dlvry": MTU.get_fully_qualified_table_name("external", "static", "qty_disc_dlvry"),
    "qty_disc_freqc": MTU.get_fully_qualified_table_name("external", "static", "qty_disc_freqc"),
    "rm_brand_vers": MTU.get_fully_qualified_table_name("oracle", "rmr", "rm_brand_vers"),
    "rm_package": MTU.get_fully_qualified_table_name("oracle", "rmr", "rm_package"),
    "rm_micro_brand_ppkg": MTU.get_fully_qualified_table_name("oracle", "rmr", "rm_micro_brand_ppkg"),
    "product": MTU.get_fully_qualified_table_name("commercial", "product", "product"),
    "promo_plan_legacy": MTU.get_fully_qualified_table_name("external", "static", "promo_plan_legacy"),
}

In [0]:
# Read oracle.rmr.rm_deal_hdr table
hdr = spark.read.table(SOURCE_TABLES["rm_deal_hdr"])

# Read oracle.rmr.rm_deal_dtl table
dtl = spark.read.table(SOURCE_TABLES["rm_deal_dtl"])

# Read oracle.rmr.rm_var_splts table
splts = spark.read.table(SOURCE_TABLES["rm_var_splts"])

# Read oracle.rmr.rm_nrp_vers table
nrp = spark.read.table(SOURCE_TABLES["rm_nrp_vers"])

# Read oracle.rmr.rm_brand_vers table
vers = spark.read.table(SOURCE_TABLES["rm_brand_vers"])

# Read oracle.rmr.rm_package table
pack = spark.read.table(SOURCE_TABLES["rm_package"])

# Read oracle.rmr.rm_micro_brand_ppkg table
ppkg = spark.read.table(SOURCE_TABLES["rm_micro_brand_ppkg"])

# Read commercial.product.product table
prod = spark.read.table(SOURCE_TABLES["product"])

# Read external.static.rmr_stat table
stat = spark.read.table(SOURCE_TABLES["rmr_stat"])

# Read external.static.rmr_actn table
actn = spark.read.table(SOURCE_TABLES["rmr_actn"])

# Read external.static.promo_cust_typ table
cust = spark.read.table(SOURCE_TABLES["promo_cust_typ"])

# Read external.static.qty_disc_dlvry table
dlvry = spark.read.table(SOURCE_TABLES["qty_disc_dlvry"])

# Read external.static.qty_disc_freqc table
freqc = spark.read.table(SOURCE_TABLES["qty_disc_freqc"])

In [0]:
hdr.createOrReplaceTempView("rm_deal_hdr")
dtl.createOrReplaceTempView("rm_deal_dtl")
splts.createOrReplaceTempView("rm_var_splts")
nrp.createOrReplaceTempView("rm_nrp_vers")
vers.createOrReplaceTempView("rm_brand_vers")
pack.createOrReplaceTempView("rm_package")
ppkg.createOrReplaceTempView("rm_micro_brand_ppkg")
prod.createOrReplaceTempView("product")
stat.createOrReplaceTempView("rmr_stat")
actn.createOrReplaceTempView("rmr_actn")
cust.createOrReplaceTempView("promo_cust_typ")
dlvry.createOrReplaceTempView("qty_disc_dlvry")
freqc.createOrReplaceTempView("qty_disc_freqc")

In [0]:
df1 = spark.sql(
    """
    SELECT 
  h.deal_hdr_seq_id, 
  DT_VAR_S.min_vs_amt, 
  DT_VAR_S.max_vs_amt 
FROM 
  rm_deal_hdr h 
  LEFT Outer Join rm_deal_dtl d on h.deal_hdr_seq_id = d.deal_hdr_seq_id 
  Left Outer Join (
    select 
      vs.deal_dtl_seq_id, 
      vs.cur_wslr_split_amt as min_vs_amt, 
      dt_vs.max_vs_amt 
    from 
      rm_var_splts vs 
      inner join (
        select 
          v.deal_dtl_seq_id, 
          sum(v.cur_wslr_split_amt) as max_vs_amt, 
          min(vs_from_amt) as min_from_amt 
        from 
          rm_var_splts as v 
        group by 
          1
      ) as DT_VS on vs.deal_dtl_seq_id = dt_vs.deal_dtl_seq_id 
    Where 
      vs.vs_from_amt = dt_vs.min_from_amt
  ) as DT_VAR_S on d.deal_dtl_seq_id = DT_VAR_S.deal_dtl_seq_id 
where 
  h.prop_beg_str_dt is null 
group by 1,2,3
"""
)

In [0]:
df1.createOrReplaceTempView("vt_var_split")

In [0]:

medallion.df = spark.sql(
    """ 
select 
  hdr.deal_hdr_seq_id as promo_plan_id, 
  extract (year from (nrp.plan_yr) ) as plan_yr_nbr, 
  nrp.state_cd as sls_dest_cd, 
  hdr.prc_chg_hdr_seq_id as frntln_prc_id, 
  rs.rmr_stat_cd, 
  hdr.mil_opt_cd, 
  hdr.pkg_grp_seq_id as pkg_grp_cd, 
  c.promo_cust_typ_cd, 
  hdr.deal_desc as promo_plan_dsc, 
  hdr.alchl_specificity_cd as alchl_spcfty_cd, 
  hdr.cnsmptn_typ_cd as premis_cnsmptn_typ_cd, 
  hdr.beg_str_dt as sls_to_rtlr_start_dt, 
  hdr.end_str_dt as sls_to_rtlr_end_dt, 
  hdr.fobr_flg, 
  hdr.beg_shipment_dt as shpmt_start_dt, 
  hdr.end_shipment_dt as shpmt_end_dt, 
  hdr.net_rvnu_amt as promo_net_rvnu_amt, 
  ra.rmr_actn_cd, 
  hdr.deal_spending_amt as promo_spnd_amt, 
  hdr.rebate_amt, 
  hdr.uniq_valdtn_flg, 
  hdr.pct_bus as rsn_tst_srce_sls_vol_pct, 
  hdr.curr_sls_vol_qty, 
  hdr.prjtd_sls_vol_qty, 
  hdr.mix_match_flg, 
  hdr.mix_match_grp_nbr, 
  hdr.var_splt_flg as varbl_split_flg, 
  hdr.qty_disc_flg, 
  dt_dd.qty_cnt as qty_disc_lvl_cnt, 
  dt_dd.min_qd_amt as qty_disc_min_disc_amt, 
  dt_dd.max_qd_amt as qty_disc_max_disc_amt, 
  qdd.qty_disc_dlvry_cd, 
  qdf.qty_disc_freqc_cd, 
  hdr.vfl_ptr_amt as promo_ver_ptr_amt, 
  hdr.vfl_ptr_eff_dt as promo_ver_ptr_eff_dt, 
  hdr.in_out_flg, 
  hdr.mulligan_flg, 
  hdr.pkg_subst_flg as pkg_ext_flg,
  hdr.pkg_subst_flg, 
  ctgy.brnd_cd,
  hdr.prc_chg_hdr_seq_id as prc_chg_reqst_id,
  cast(hdr.aprvl_dt as date) as promo_aprvl_dt, 
  vsplit.min_vs_amt as min_wslr_split_amt, 
  vsplit.max_vs_amt as max_wslr_split_amt, 
  hdr.wslr_number as brwy_cust_nbr,
  (case when hdr.rebate_amt is not null and hdr.rebate_amt > 0 then 'Y' else 'N' end) as rebate_flg,
  (case when dt_bv.type_cd <> 'm' then dt_bv.alpha_cd else vt_brand_vers.alpha_cd end) as alpha_cd,
  hdr.__created_tsp as edw_start_tsp,
  hdr.__created_tsp 
  from rm_deal_hdr hdr 
  inner join rm_nrp_vers nrp on hdr.nrp_vers_seq_id = nrp.nrp_vers_seq_id 
  /* ri for stat_cd*/
  left outer join rmr_stat rs on hdr.stat_cd = rs.rmr_stat_cd 
  /* ri for action cd */
  left outer join rmr_actn ra on hdr.action_cd = ra.rmr_actn_cd 
  left outer join (
    select 
      dtl.deal_hdr_seq_id, 
      count(*) as qty_cnt, 
      min(dtl.curr_max_disc_amt) as min_qd_amt, 
      max(dtl.curr_max_disc_amt) as max_qd_amt 
    from 
      rm_deal_dtl dtl 
    group by 
      dtl.deal_hdr_seq_id
  ) as dt_dd on hdr.deal_hdr_seq_id = dt_dd.deal_hdr_seq_id 
  /*ri for wslr cust grp typ cd */
  left outer join promo_cust_typ c on hdr.wslr_cust_grp_typ_cd = c.promo_cust_typ_cd 
  /* ri qty disc dlvry */
  left outer join qty_disc_dlvry qdd on hdr.qd_delivery_cd = qdd.qty_disc_dlvry_cd 
  /*ri for qty_disc_freqc_cd */
  left outer join qty_disc_freqc qdf on hdr.qd_frequency_cd = qdf.qty_disc_freqc_cd 
  left outer join vt_var_split vsplit on hdr.deal_hdr_seq_id = vsplit.deal_hdr_seq_id 
  left outer join (
    select brand_seq_id, alpha_cd, type_cd 
    from 
      rm_brand_vers qualify row_number() over(
        partition by brand_seq_id 
        order by 
          vers_nbr desc, 
          __created_tsp desc
      ) = 1
  ) as dt_bv on hdr.brand_seq_id = dt_bv.brand_seq_id
  left outer join 
 (select mb.brand_cd, rbv.alpha_cd, rbv.type_cd, rbv.brand_seq_id 
                                  from  rm_package rp
                               inner join  
                              ( select brand_seq_id, brand_cd,package_id as max_pkg_id
                                 from  rm_micro_brand_ppkg
                               qualify row_number() over(partition by brand_seq_id,brand_cd order by vers_nbr desc, __created_tsp desc) = 1
                               ) as mb   
                                    on rp.package_id = mb.max_pkg_id
                                    inner join
                                (select brand_seq_id, alpha_cd, type_cd
                                   from rm_brand_vers
                                qualify row_number() over(partition by brand_seq_id  order by __created_tsp desc,vers_nbr desc) = 1
                                ) rbv
                                   on mb.brand_seq_id = rbv.brand_seq_id 
                                   
                                   group by 1,2,3,4
 ) as vt_brand_vers
on dt_bv.alpha_cd = vt_brand_vers.brand_cd
and dt_bv.type_cd = 'm'
 left outer join  
        ( select brnd_cd
          from  product       
          group by 1 
         ) as ctgy  
       on (case when dt_bv.type_cd <> 'm' then dt_bv.alpha_cd else vt_brand_vers.alpha_cd end) = ctgy.brnd_cd 
"""
)

In [0]:
# if the table exists run the scd type 2 update
if spark.catalog.tableExists(medallion.table_lookup["gold"]):
    try:  # Transform and select final columns
        medallion.df = (
            medallion.df.withColumn("__created_tsp", F.lit(medallion.start_datetime).cast(TimestampType()))
            .withColumn("edw_start_tsp", F.expr("__created_tsp"))
            .withColumn("edw_end_tsp", F.lit(None))
        )
        medallion.df = medallion.df.select(
            *[
                (F.col(x.name).cast(x.dataType))
                for x in spark.read.table(SOURCE_TABLES["promo_plan_legacy"]).schema.fields
            ]
        )
        # write gold table
        medallion.write_to_delta(load_type="scd_type_2", layer="gold")
        medallion.logger.info("write to gold started for internal_order using SCD TYPE 2 with source data")
    except Exception as e:
        medallion.logger.error(f"Error writing internal_order table to gold layer. Error message: {e}")
        raise

# otherwise do historical backfill
else:
    hist_df = spark.read.table(SOURCE_TABLES["promo_plan_legacy"])
    try:
        medallion.df = hist_df.withColumn("__created_tsp", F.lit(medallion.start_datetime).cast(TimestampType()))
        # write gold table
        medallion.write_to_delta(load_type="overwrite", layer="gold")
        medallion.logger.info("write to gold started for internal_order using OVERWRITE with historical backfill")
    except Exception as e:
        medallion.logger.error(f"Error writing internal_order table to gold layer. Error message: {e}")
        raise

In [0]:
# publish to external stage - snowflake
try:
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()

except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
        config.get("comparison_check", {}).get("treat_nulls"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )